A Second, more cooler demo, of the career agent

In [ ]:
# Required Imports

from google import genai
from google.genai import types
import fitz
import docx
import time

In [ ]:
#Note: if using a full account, look into authenticating using Application Default Credentials.

# ── Configuration ────────────────────────────────────────────────────────────
API_KEY = "INSERT_API_KEY_HERE"
MODEL = "gemini-2.5-flash"
 
client = genai.Client(vertexai=True, api_key=API_KEY)

# Express mode has lower rate limits, so we add a short pause between
# back-to-back API calls to avoid hitting them.
DELAY_BETWEEN_CALLS = 2  # seconds

In [ ]:
# ── Pretty Logging (makes the agentic loop visible) ──────────
 
class AgentLogger:
    """Prints structured, colour-coded logs so the audience can follow
    every step of the agentic loop in real time."""
 
    HEADER = "\033[95m"
    BLUE = "\033[94m"
    CYAN = "\033[96m"
    GREEN = "\033[92m"
    YELLOW = "\033[93m"
    RED = "\033[91m"
    BOLD = "\033[1m"
    DIM = "\033[2m"
    RESET = "\033[0m"
 
    @staticmethod
    def banner(text: str):
        width = 64
        print(f"\n{AgentLogger.BOLD}{AgentLogger.HEADER}")
        print("═" * width)
        print(f"  {text}")
        print("═" * width)
        print(AgentLogger.RESET)
 
    @staticmethod
    def step(icon: str, label: str, detail: str = ""):
        prefix = f"{AgentLogger.BOLD}{AgentLogger.CYAN}{icon}{AgentLogger.RESET}"
        label_str = f"{AgentLogger.BOLD}{label}{AgentLogger.RESET}"
        print(f"  {prefix}  {label_str}  {AgentLogger.DIM}{detail}{AgentLogger.RESET}")
 
    @staticmethod
    def tool_call(name: str, args_summary: str):
        print(f"\n  {AgentLogger.YELLOW}{AgentLogger.BOLD}🔧 TOOL CALL:{AgentLogger.RESET} "
              f"{AgentLogger.BOLD}{name}{AgentLogger.RESET}")
        print(f"     {AgentLogger.DIM}args: {args_summary}{AgentLogger.RESET}")
 
    @staticmethod
    def tool_result(name: str, preview: str):
        print(f"  {AgentLogger.GREEN}✅ TOOL RESULT:{AgentLogger.RESET} {name}")
        # Show first 200 chars of the result as a preview
        truncated = preview[:200] + ("…" if len(preview) > 200 else "")
        for line in truncated.split("\n"):
            print(f"     {AgentLogger.DIM}{line}{AgentLogger.RESET}")
 
    @staticmethod
    def round_header(round_num: int):
        print(f"\n  {AgentLogger.BLUE}{AgentLogger.BOLD}"
              f"── Round {round_num} ──────────────────────────────────"
              f"{AgentLogger.RESET}")
 
    @staticmethod
    def final():
        print(f"\n  {AgentLogger.GREEN}{AgentLogger.BOLD}"
              f"✅ Agent finished — model returned final text response."
              f"{AgentLogger.RESET}\n")
 
    @staticmethod
    def timing(label: str, seconds: float):
        print(f"     {AgentLogger.DIM}⏱  {label}: {seconds:.1f}s{AgentLogger.RESET}")

    @staticmethod
    def warning(text: str):
        print(f"  {AgentLogger.YELLOW}⚠️  {text}{AgentLogger.RESET}")

    @staticmethod
    def retry(attempt: int, max_attempts: int, reason: str):
        print(f"  {AgentLogger.RED}🔄 Retry {attempt}/{max_attempts}: "
            f"{reason}{AgentLogger.RESET}")
 

log = AgentLogger()

In [4]:
# ── Resilient LLM Call Helper ────────────────────────────────────────────────
#
# Express mode can return empty responses due to rate limits or transient
# issues.  This wrapper retries with exponential backoff so the demo
# doesn't crash on stage.
 
MAX_RETRIES = 3
 
 
def _call_model_with_retry(*, model: str, contents, config=None,
                           call_label: str = "LLM call") -> object:
    """
    Calls client.models.generate_content with retry logic.
    Returns a response object guaranteed to have non-empty content.
    Raises RuntimeError only after all retries are exhausted.
    """
    for attempt in range(1, MAX_RETRIES + 1):
        if attempt > 1:
            wait = DELAY_BETWEEN_CALLS * attempt  # exponential-ish backoff
            log.retry(attempt, MAX_RETRIES,
                      f"waiting {wait}s before retrying {call_label}")
            time.sleep(wait)
 
        t0 = time.time()
        try:
            if config:
                response = client.models.generate_content(
                    model=model, contents=contents, config=config,
                )
            else:
                response = client.models.generate_content(
                    model=model, contents=contents,
                )
        except Exception as e:
            log.warning(f"{call_label} raised an exception: {e}")
            if attempt == MAX_RETRIES:
                raise RuntimeError(
                    f"{call_label} failed after {MAX_RETRIES} attempts: {e}"
                ) from e
            continue
 
        elapsed = time.time() - t0
        log.timing(call_label, elapsed)
 
        # ── Validate the response ────────────────────────────────────────
        if not response.candidates:
            log.warning(f"{call_label}: no candidates returned")
            if attempt == MAX_RETRIES:
                raise RuntimeError(
                    f"{call_label}: no candidates after {MAX_RETRIES} attempts. "
                    "The request may have been blocked."
                )
            continue
 
        candidate = response.candidates[0]
 
        if candidate.content is None:
            reason = getattr(candidate, "finish_reason", "UNKNOWN")
            log.warning(f"{call_label}: content is None (finish_reason={reason})")
            if attempt == MAX_RETRIES:
                raise RuntimeError(
                    f"{call_label}: content is None after {MAX_RETRIES} attempts. "
                    f"Finish reason: {reason}"
                )
            continue
 
        if not candidate.content.parts:
            reason = getattr(candidate, "finish_reason", "UNKNOWN")
            log.warning(f"{call_label}: empty parts (finish_reason={reason})")
            if attempt == MAX_RETRIES:
                if "MALFORMED_FUNCTION_CALL" in str(reason):
                    raise RuntimeError(
                        f"{call_label}: model produced a malformed tool call after "
                        f"{MAX_RETRIES} attempts. Try upgrading the model — "
                        "'gemini-2.5-flash' handles function calling more reliably "
                        "than 'gemini-2.5-flash-lite'."
                    )
                raise RuntimeError(
                    f"{call_label}: empty response parts after {MAX_RETRIES} "
                    f"attempts. Finish reason: {reason}. "
                    "This can happen with Express mode rate limits — "
                    "try again in a few seconds."
                )
            continue
 
        # All checks passed — valid response
        return response
 
    # Should never reach here, but just in case:
    raise RuntimeError(f"{call_label}: exhausted all retries")


In [5]:
# ── Tool Declarations (schemas the orchestrator sees) ────────────────────────
 
analyze_tool = types.FunctionDeclaration(
    name="analyze_resume_against_jd",
    description=(
        "Analyzes a resume against a job description. "
        "Returns specific, actionable restructuring advice including: "
        "keyword gaps, section reordering suggestions, bullet point rewrites, "
        "and quantification opportunities."
    ),
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "resume_text": types.Schema(
                type="STRING", description="Full text of the resume"
            ),
            "job_description": types.Schema(
                type="STRING", description="Full text of the job description"
            ),
        },
        required=["resume_text", "job_description"],
    ),
)
 
cover_letter_tool = types.FunctionDeclaration(
    name="draft_cover_letter",
    description=(
        "Drafts a tailored cover letter or application email. "
        "Call this when the user asks to draft, write, or create a cover letter, "
        "application email, or application letter."
    ),
    parameters=types.Schema(
        type="OBJECT",
        properties={
            "resume_text": types.Schema(
                type="STRING", description="Full text of the resume"
            ),
            "job_description": types.Schema(
                type="STRING", description="Full text of the job description"
            ),
            "tone": types.Schema(
                type="STRING",
                enum=["formal", "conversational", "enthusiastic"],
                description="Desired tone of the cover letter (default: formal)",
            ),
        },
        required=["resume_text", "job_description"],
    ),
)
 
tools = types.Tool(function_declarations=[analyze_tool, cover_letter_tool])
 
SYSTEM_PROMPT = """You are a professional career coach and resume expert.
 
You have access to two tools — ALWAYS use them rather than answering directly:
  • analyze_resume_against_jd — call when asked to review, analyze, or improve a resume.
  • draft_cover_letter — call when asked to write a cover letter or application email.
 
If the user asks for BOTH, call analyze_resume_against_jd first, then draft_cover_letter.
 
When you receive the tool results, synthesize them into a polished, encouraging,
and actionable response for the user.  Structure your response with clear sections
and preserve all specific advice from the tools."""


In [ ]:
# ── Tool Execution (each tool makes its own focused LLM call) ────────────────
 
def execute_tool(name: str, args: dict) -> str:
    """
    Execute a tool by making a focused inner LLM call.
    Each tool has its own specialised prompt that produces
    detailed, structured output.
    """
 
    if name == "analyze_resume_against_jd":
        return _run_resume_analysis(
            resume_text=args.get("resume_text", ""),
            jd_text=args.get("job_description", ""),
        )
 
    elif name == "draft_cover_letter":
        return _run_cover_letter_draft(
            resume_text=args.get("resume_text", ""),
            jd_text=args.get("job_description", ""),
            tone=args.get("tone", "formal"),
        )
 
    return f"Unknown tool: {name}"
 
 
def _run_resume_analysis(resume_text: str, jd_text: str) -> str:
    """Inner LLM call: deep resume-vs-JD analysis."""
 
    log.step("🧠", "Inner LLM call", "resume analysis specialist")
 
    analysis_prompt = f"""You are a senior technical recruiter and resume strategist.
 
Analyze the following resume against the job description.  Provide a
DETAILED and SPECIFIC report with these exact sections:
 
1. **KEYWORD GAP ANALYSIS**
   List specific keywords/phrases from the JD that are missing or
   underrepresented in the resume.  For each, suggest where and how
   to incorporate it.
 
2. **SECTION-BY-SECTION FEEDBACK**
   For each resume section (Summary, Experience, Skills, Education, etc.),
   give specific improvement advice.
 
3. **BULLET POINT REWRITES**
   Pick 2-3 weak bullet points and rewrite them using the STAR method
   with quantified impact.
 
4. **OVERALL MATCH SCORE**
   Rate the resume's alignment to the JD on a scale of 1-10 with
   justification.
 
===== RESUME =====
{resume_text}
 
===== JOB DESCRIPTION =====
{jd_text}
"""
 
    # Pause before inner call to avoid back-to-back rate limit hits
    time.sleep(DELAY_BETWEEN_CALLS)
 
    response = _call_model_with_retry(
        model=MODEL,
        contents=analysis_prompt,
        call_label="Resume analysis (inner)",
    )
    return response.text
 
 
def _run_cover_letter_draft(resume_text: str, jd_text: str, tone: str) -> str:
    """Inner LLM call: tailored cover letter generation."""
 
    log.step("🧠", "Inner LLM call", f"cover letter specialist (tone: {tone})")
 
    tone_guidance = {
        "formal": "Use a professional, polished tone. Avoid slang or overly casual language.",
        "conversational": "Use a warm, approachable tone — like a confident professional chatting over coffee.",
        "enthusiastic": "Use an energetic, passionate tone that conveys genuine excitement for the role.",
    }
 
    draft_prompt = f"""You are an expert career writer who crafts compelling cover letters.
 
Write a complete, ready-to-send cover letter using the resume and job
description below.
 
REQUIREMENTS:
- {tone_guidance.get(tone, tone_guidance["formal"])}
- Open with a strong hook — not "I am writing to apply for…"
- Map 2-3 specific achievements from the resume to requirements in the JD.
- Show genuine knowledge of the company and role.
- Close with a confident call to action.
- Keep it to roughly 300-400 words.
- Format it as a proper letter with [Your Name] as placeholder.
 
===== RESUME =====
{resume_text}
 
===== JOB DESCRIPTION =====
{jd_text}
"""
 
    # Pause before inner call to avoid back-to-back rate limit hits
    time.sleep(DELAY_BETWEEN_CALLS)
 
    response = _call_model_with_retry(
        model=MODEL,
        contents=draft_prompt,
        call_label="Cover letter draft (inner)",
    )
    return response.text

In [7]:
# ── Document Loader ─────────────────────────────────────────────────────────
 
def load_document(path: str) -> str:
    """Extract text from .pdf, .docx, or .txt files."""
    if path.endswith(".pdf"):
        text = ""
        with fitz.open(path) as pdf:
            for page in pdf:
                text += page.get_text()
        return text.strip()
 
    if path.endswith(".docx"):
        doc = docx.Document(path)
        return "\n".join(p.text for p in doc.paragraphs).strip()
 
    if path.endswith(".txt"):
        with open(path, encoding="utf-8") as f:
            return f.read().strip()
 
    raise ValueError(f"Unsupported file type: {path}")


In [8]:
# ── Agentic Loop ────────────────────────────────────────────────────────────
 
MAX_TOOL_ROUNDS = 5  # safety limit to avoid infinite loops
 
 
def run_career_agent(user_message: str, resume_path: str, jd_path: str) -> str:
    """
    The main agentic loop:
      1. Send the user's request + documents to the orchestrator model.
      2. If the model returns function calls → execute each tool
         (which makes its own inner LLM call) → feed results back.
      3. Repeat until the model returns a final text response.
    """
 
    log.banner("CAREER COACHING AGENT — Starting")
 
    # ── Load documents ───────────────────────────────────────────────────
    log.step("📄", "Loading resume", resume_path)
    resume_text = load_document(resume_path)
    log.step("📄", "Loading job description", jd_path)
    jd_text = load_document(jd_path)
    log.step("📏", "Document sizes",
             f"Resume: {len(resume_text)} chars | JD: {len(jd_text)} chars")
 
    # ── Build the generation config ──────────────────────────────────────
    gen_config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        tools=[tools],
        tool_config=types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode="AUTO",
            )
        ),
    )
 
    # ── Initial message ──────────────────────────────────────────────────
    log.step("💬", "User request", user_message)
 
    messages = [
        types.Content(
            role="user",
            parts=[
                types.Part(text=f"Here is my resume:\n\n{resume_text}"),
                types.Part(text=f"Here is the job description:\n\n{jd_text}"),
                types.Part(text=user_message),
            ],
        )
    ]
 
    # ── Loop until text response or limit reached ────────────────────────
    total_start = time.time()
 
    for round_num in range(1, MAX_TOOL_ROUNDS + 1):
        log.round_header(round_num)
        log.step("🤖", "Calling orchestrator model", MODEL)
 
        # Use the retry wrapper for every orchestrator call
        response = _call_model_with_retry(
            model=MODEL,
            contents=messages,
            config=gen_config,
            call_label=f"Orchestrator (round {round_num})",
        )
 
        candidate = response.candidates[0]
 
        # — Collect any function calls ------------------------------------
        function_calls = [p for p in candidate.content.parts if p.function_call]
 
        if not function_calls:
            # No tool calls → the model produced its final text answer.
            log.final()
            log.timing("Total agent time", time.time() - total_start)
            return response.text
 
        # — Log what the orchestrator decided to do -----------------------
        log.step("🧭", "Orchestrator decision",
                 f"calling {len(function_calls)} tool(s): "
                 f"{', '.join(fc.function_call.name for fc in function_calls)}")
 
        # — Execute every requested tool ----------------------------------
        messages.append(candidate.content)
 
        tool_response_parts = []
        for fc_part in function_calls:
            fn_name = fc_part.function_call.name
            fn_args = dict(fc_part.function_call.args)
 
            # Summarise args for logging (without dumping full text)
            args_summary = ", ".join(
                f"{k}={'<' + str(len(str(v))) + ' chars>'}"
                if len(str(v)) > 50 else f"{k}={v!r}"
                for k, v in fn_args.items()
            )
            log.tool_call(fn_name, args_summary)
 
            # Inject full resume/JD so the tool has the source material
            fn_args["resume_text"] = resume_text
            fn_args["job_description"] = jd_text
 
            # Execute the tool (this makes an inner LLM call with retry)
            tool_result = execute_tool(fn_name, fn_args)
            log.tool_result(fn_name, tool_result)
 
            tool_response_parts.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        name=fn_name,
                        response={"result": tool_result},
                    )
                )
            )
 
        # Send all tool responses back in one user turn
        messages.append(
            types.Content(role="user", parts=tool_response_parts)
        )
        log.step("📨", "Sent tool results back to orchestrator",
                 f"{len(tool_response_parts)} result(s)")
 
        # Brief pause before next orchestrator call
        time.sleep(DELAY_BETWEEN_CALLS)
 
    # If we exhaust all rounds without a text answer, return what we have.
    log.timing("Total agent time", time.time() - total_start)
    return response.text


In [9]:
# ── Main ─────────────────────────────────────────────────────────────────────

result = run_career_agent(
    user_message=(
        "Please analyze my resume against this job description, "
        "and then draft a formal cover letter for the same role."
    ),
    resume_path="./sample_resume.pdf",
    jd_path="./jd.txt",
)

print("\n")
log.banner("FINAL AGENT RESPONSE")
print(result)




════════════════════════════════════════════════════════════════
  CAREER COACHING AGENT — Starting
════════════════════════════════════════════════════════════════

  📄  Loading resume  ./sample_resume.pdf
  📄  Loading job description  ./jd.txt
  📏  Document sizes  Resume: 2138 chars | JD: 2938 chars
  💬  User request  Please analyze my resume against this job description, and then draft a formal cover letter for the same role.

  ── Round 1 ──────────────────────────────────
  🤖  Calling orchestrator model  gemini-2.5-flash
     ⏱  Orchestrator (round 1): 16.7s
  🧭  Orchestrator decision  calling 1 tool(s): analyze_resume_against_jd

  🔧 TOOL CALL: analyze_resume_against_jd
     args: resume_text=<2138 chars>, job_description=<2938 chars>
  🧠  Inner LLM call  resume analysis specialist
     ⏱  Resume analysis (inner): 20.9s
  ✅ TOOL RESULT: analyze_resume_against_jd
     Here's a detailed analysis of Liam Sterling's resume against the Junior Python Developer job description at Rain